##### Testing Sampling Against Papers
This will be used to evaluate the performance of the package against reference peer reviewed papers.

!["alt"](../images/franks2014Table3.png "Franks2014 p848 Table3")

In [ ]:
import networkx as nx
from Influence_Modelling import XsvNetwork, NetworkMetrics, Sampling

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
enron = XsvNetwork.XsvNetwork("", "", "enron")
enron_nx = enron.get_nx()

airports = XsvNetwork.XsvNetwork("", "", "airports")
airports_nx = airports.get_nx()

twitter = XsvNetwork.XsvNetwork("", "", "twitter")
twitter_nx = twitter.get_nx()

p2p = XsvNetwork.XsvNetwork("", "", "p2p")
p2p_nx = p2p.get_nx()

enron_nx = enron.get_nx()
airports_nx = airports.get_nx()
twitter_nx = twitter.get_nx()
p2p_nx = p2p.get_nx()

The below function will be used to generate the data that is held in the rows of the above table.

In [53]:
def generate_row(input_graph: nx.DiGraph):
  return pd.DataFrame(data = {
   'nodes': [NetworkMetrics.s_1_number_nodes(input_graph)], 
    'edges': [NetworkMetrics.s_2_number_edges(input_graph)],
    'mean_degree': [NetworkMetrics.s_3_mean_degree(input_graph)],
    'cc': [NetworkMetrics.s_4_clustering_coefficient(input_graph)],
    #'centralization': [NetworkMetrics.s_6_centralization(input_graph)]  
  })

generate_row(enron_nx)

,nodes,edges,mean_degree,cc
0,36692,183831,10.020222,0.496983


In [ ]:
table = []
for name, data in [("enron", enron_nx), ("p2p",p2p_nx), ("airport", airports_nx)]:
  table.append([name]+generate_row(data))
print(table)
raw_data = pd.DataFrame(table, columns=["Dataset", "Nodes", "Edges", "Mean Degree", "Clustering Coefficient", "Centralization"])
raw_data

Compare the sampling methods on the enron dataset as an exmaple and generalise to allow for any sample

In [ ]:
def sampled_comparison(input_data: nx.DiGraph):
  sampled_datasets = {
    "bfs": Sampling.bfs_sample(input_data, Sampling.random_sample([n for n in input_data.nodes()], 1), 1000),
   # "sns": Sampling.snowball_sample(input_data, Sampling.random_sample([n for n in input_data.nodes()], 1), 1000),
    "mhrw": Sampling.mhrw_sample(input_data, 1000),
    "mhda": Sampling.mhda_sample(input_data, 10)
  }
  return sampled_datasets

samples = sampled_comparison(enron_nx)
print(generate_row(samples['mhda']))

In [ ]:
table = []
for method, data in [(method, generate_row(samples[method])) for method in ['bfs', 'mhrw', 'mhda']]:
  table.append([method]+data)
raw_data = pd.DataFrame(table, columns=["Dataset", "Nodes", "Edges", "Mean Degree", "Clustering Coefficient", "Centralization"])
raw_data

In [55]:
import numpy as np
import pandas as pd

def multiple_runs(G: nx.DiGraph, label: str, sampling_method, trials: int = 10, nodes: int = 1000):
    results = pd.DataFrame()
    for _ in range(trials):
        sample: nx.DiGraph = sampling_method(G, nodes)
        results = pd.concat([results, generate_row(sample)])
    df = pd.DataFrame(
        results, columns=["nodes", "edges", "mean_degree", "cc"]#, "centralization"]
    )
    return pd.DataFrame(data = 
        {
            'method': [label],
            'nodes': [df['nodes'].to_numpy().mean()],
            'nodes_std': [df['nodes'].to_numpy().std()],
            'edges': [df['edges'].to_numpy().mean()],
            'edges_std': [df['edges'].to_numpy().std()],
            'mean_degree': [df['mean_degree'].to_numpy().mean()],
            'mean_degree_std':[ df['mean_degree'].to_numpy().std()],
            'cc':[ df['cc'].to_numpy().mean()],
            'cc_std': [df['cc'].to_numpy().std()],
            #'centralization': [df['centralization'].to_numpy().mean()],
            #'centralization_std': [df['centralization'].to_numpy().std()],
         }
    )

def generate_table_3_2014_learning():
    trials = 5
    nodes = 10
    datasets = [(enron_nx, "enron"), (p2p_nx, "p2p")] #... etc
    methods = [(Sampling.mhrw_sample, "MHRW"), (Sampling.mhda_sample, "MHDA")]#, (Sampling.snowball_sample, "SNS")] etc
    results = []
    print("Generating Table 3 Franks 2014 Learning Paper")
    for data, label in datasets:
        print(f"Started {label} Table -----------")
        method_runs = []
        for method, method_name in methods:
            print(f"Started {label}-{method_name} Sampling")
            method_runs.append(multiple_runs(data, method_name, method, trials, nodes))
            print(f"Completed {label}-{method_name} Sampling")
        table: pd.DataFrame = pd.concat(method_runs)
        results.append(table)
        print(f"Completed {label} Table -----------")
    return results

enron_sampled = generate_table_3_2014_learning()

Generating Table 3 Franks 2014 Learning Paper
Started enron Table -----------
Started enron-MHRW Sampling
Completed enron-MHRW Sampling
Started enron-MHDA Sampling
Completed enron-MHDA Sampling
Completed enron Table -----------
Started p2p Table -----------
Started p2p-MHRW Sampling
Completed p2p-MHRW Sampling
Started p2p-MHDA Sampling
Completed p2p-MHDA Sampling
Completed p2p Table -----------


In [56]:
enron_sampled[0]

,method,nodes,nodes_std,edges,edges_std,mean_degree,mean_degree_std,cc,cc_std
0,MHRW,10.0,0.0,12.6,2.154066,2.52,0.430813,0.320762,0.189595
0,MHDA,10.0,0.0,7.0,3.577709,1.40,0.715542,0.077619,0.155238
